In [1]:
FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'
#FILE_PATH = 'D://bitmex_data_1m.csv'

from abc import ABC, abstractmethod
from common import *
import plotly
import pandas as pd

In [2]:
test_df = pd.read_csv(FILE_PATH, delimiter=',')
test_df

,timestamp,symbol,open,high,low,close,trades,volume,vwap,lastSize,turnover,homeNotional,foreignNotional
0,2015-09-25 12:01:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
1,2015-09-25 12:02:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
2,2015-09-25 12:03:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
3,2015-09-25 12:04:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
4,2015-09-25 12:05:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4005552,2023-05-08 03:13:00+00:00,XBTUSD,28320.5,28328.5,28320.5,28328.5,12,50000,28322.75,200.0,176536630,1.765366,50000.0
4005553,2023-05-08 03:14:00+00:00,XBTUSD,28328.5,28330.0,28329.5,28330.0,11,43500,28329.65,200.0,153549633,1.535496,43500.0
4005554,2023-05-08 03:15:00+00:00,XBTUSD,28330.0,28329.5,28320.0,28320.0,51,173600,28326.10,100.0,612862282,6.128623,173600.0
4005555,2023-05-08 03:16:00+00:00,XBTUSD,28320.0,28322.5,28289.5,28304.0,154,1151700,28313.05,3300.0,4067744000,40.677440,1151700.0


In [3]:
import numpy as np

In [19]:


# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str = None):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        
        if(self.from_date):
            df = df.loc[df.timestamp >= self.from_date]  #일단 GMT 니까 1분 뒤로 조정할 걸 생각하고 +1분부터 가져오면 된다.
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        data['ma_20']=self.__sma(data,20)
        data['ma_60']=self.__sma(data,60)
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __sma(self, data, period=20):
        return data['close'].rolling(window=period, min_periods=1).mean()
    
# 구체적인 데이터 처리 전략: RSI 계산
class RSIProcessing(DataProcessingStrategy):
    def process_data(self, data):
        data['RSI'] = self.__rsi(data)
        print("RSI를 계산했습니다.")
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __rsi(self, data, period=14):
    
        import numpy as np
        
        delta = data['close'].diff(1)  # 종가의 변화량 계산
        gain = np.where(delta > 0, delta, 0)  # 상승분
        loss = np.where(delta < 0, -delta, 0)  # 하락분
    
        avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
        avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()
        
        rs = avg_gain / (avg_loss + 1e-10)  # 0으로 나누는 오류 방지
        rsi = 100 - (100 / (1 + rs))
        
        rsi.index = data.index
        
        return rsi

#고점을 찾는 처리 전략
class HighPointScoringProcessing(DataProcessingStrategy):
    '''
    메인 df에 'high_score' 라는 컬럼을 추가하고, 외부 변수의 리스트로 들어온 
    밴드 값을 shift 하면서 그중에 가장 큰 가격에 +1 스코어를 한다.
    '''
    
    
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
    
    def process_data(self, data):
        data = self.__get_high_score_by_list(data,self.bandwith_list)
        return data
    
    def __get_high_score_by_list(self, origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_high_score_with_bandwidth(origin_df, 'high', i)
                else:
                    return_df = self.__get_high_score_with_bandwidth(return_df, 'high', i, reset=False)
    
        return return_df
    
    
        
    def __get_high_score_with_bandwidth(self, target_df, column_name, bandwidth, reset=True):
        '''
        df를 제공하면서 밴드 값을 같이 제공하면 이를 반복문으로 돌아가면서 score 를 쌓는 함수
        '''
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['high_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            max_index = target_df.iloc[i:i+bandwidth][column_name].idxmax()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
        #
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
        #
            #print(f"체크값 :{target_df.loc[max_index]['high_score']+1}")
            #
            target_df.loc[max_index,'high_score'] = target_df.loc[max_index]['high_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
        
        print(f"수행한 숫자:{end_index}")
        print(f"가장 높은 점수:{target_df.loc[target_df['high_score'].idxmax()]['high_score']}")
        
        return target_df

#저점을 찾는 처리 전략
class LowPointScoringProcessing(DataProcessingStrategy):
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
        
    def process_data(self, data):
        data = self.__get_low_score_by_list(data, self.bandwith_list)
        return data
    
    def __get_low_score_by_list(self,origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_low_score_with_bandwidth(origin_df, 'low', i)
                else:
                    return_df = self.__get_low_score_with_bandwidth(return_df, 'low', i, reset=False)
        
        return return_df
    
    
    def __get_low_score_with_bandwidth(self,target_df, column_name, bandwidth, reset=True):
    
    
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['low_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            min_index = target_df.iloc[i:i+bandwidth][column_name].idxmin()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[min_index,'low_score'] = target_df.loc[min_index]['low_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            

        return target_df

# 고점만을 필터팅하는 처리 전략
class GetHighPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['high_score']!=0]
        final_high_point=self.__apply_threshold_with_normalizing_for_high_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_high_point
        
    #df와 타겟 컬럼명을 받아서 해당 %이상의 값만 필터링하는 함수
    def __apply_threshold_with_normalizing_for_high_value(self, target_df, target_column, threshold):
    
        #min-max 정규화
        target_df['normalized_value_high'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_high'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #상위 5%에 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_high'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'high_for_graph'] = df_calculated['high']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_high'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_high_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['high_score'] > base_score]
        df_calculated.loc[df_calculated['high_score'] > base_score, 'origin_score'] = df_calculated['high_score']
        df_calculated.loc[df_calculated['high_score'] > base_score, 'high_score'] = df_calculated['high']
    
        import numpy as np
        df_calculated.loc[df_calculated['high_score'] <= base_score, 'high_score'] = np.nan
        
        return df_calculated
    
# 저점만을 필터팅하는 처리 전략
class GetLowPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['low_score']!=0]
        final_low_point=self.__apply_threshold_with_normalizing_for_low_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_low_point
    
    def __apply_threshold_with_normalizing_for_low_value(self, target_df, target_column, threshold):
        #min-max 정규화
        target_df['normalized_value_low'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_low'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #threshold 로 적은 수 이하로 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_low'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'origin_low_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_low'] >= threshold, 'low_for_graph'] = df_calculated['low']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_low'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_low_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['low_score'] > base_score]
        df_calculated.loc[df_calculated['low_score'] > base_score, 'origin_score'] = df_calculated['low_score']
        df_calculated.loc[df_calculated['low_score'] > base_score, 'low_score'] = df_calculated['low']
    
        import numpy as np
        df_calculated.loc[df_calculated['low_score'] <= base_score, 'low_score'] = np.nan
        
        return df_calculated
    
    
# 데이터 시각화 인터페이스
class Visualization(ABC):
    @abstractmethod
    def visualize(self):
        pass

    def add_trace(self, trace):
        self.price_trace_list.append(trace)
    

# 구체적인 시각화 전략: 라인 그래프
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class BasicPriceWithRsiVisualization(Visualization):
    
    def __init__(self):
        self.price_trace_list = []
        self.rsi_trace_lsit = []
        
    def set_data(self, df, high_points_df, low_points_df):
        self.__df = df
        self.__high_points_df = high_points_df
        self.__low_points_df = low_points_df
        
    def visualize(self):
        self.__get_base_figure()
        self.__main_figure.show()
        
        
        
    def __get_base_figure(self):
        main_trace = go.Candlestick(
            x=list(self.__df['timestamp_kst']),
            open=list(self.__df['open']),
            high=list(self.__df['high']),
            low=list(self.__df['low']),
            close=list(self.__df['close']),
        )
        
        #두번째 고가 점 트레이스 만들기
        high_point_trace = go.Scatter(y=list(self.__high_points_df['high_for_graph']), x=list(self.__high_points_df['timestamp_kst']), 
        marker=dict(
        color='rgba(255, 0, 255, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='고점 그래프')
        
        #세번째 저가 점 트레이스 만들기
        low_point_trace = go.Scatter(y=list(self.__low_points_df['low_for_graph']), x=list(self.__low_points_df['timestamp_kst']), 
        marker=dict(
        color='rgba(0, 0, 0, 1)',  # 점의 색상 설정 (RGBa 형식)
        size=5,  # 점의 크기 설정
        ),
        mode='markers', name='저점 그래프')
        
        ma_20_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_20'], mode='lines', name='MA_20', line=dict(color='blue'))

        ma_60_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['ma_60'], mode='lines', name='MA_60', line=dict(color='black'))
        
        rsi_trace = go.Scatter(x=self.__df['timestamp_kst'], y=self.__df['RSI'], mode='lines', name='RSI', line=dict(color='blue'))
        
        main_figure = self.__draw_subplots([main_trace,high_point_trace,low_point_trace,ma_20_trace,ma_60_trace],[rsi_trace])
        
        self.__main_figure = main_figure
   
    
    def __draw_subplots(self, price_trace_list, rsi_trace_list):
        
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        row_heights=[0.7, 0.3],  # 위쪽(가격) 70%, 아래쪽(RSI) 30%
                        subplot_titles=("가격 차트", "RSI (14)"))
        
        fig.update_layout(
        title='Candlestick Chart',
        xaxis_title='Date',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        width=1000,  # 그래프 너비 설정
        height=800   # 그래프 높이 설정
        )
        
        #trace_list1를 전부 담음
        for i in price_trace_list:
            fig.add_trace(i, row=1, col=1)
        
        for i in rsi_trace_list:
            fig.add_trace(i, row=2, col=1)
            fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought (70)", row=2, col=1)
            fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold (30)", row=2, col=1)
            
            
        return fig

        

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def add_sub_indicator(self,indicator_instance_list):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.'''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
            
    def get_key_points(self, instance : DataProcessingStrategy):
        '''고점을 찾거나 다이버전스를 찾는등의 주요 포인트를 찾을 때 활용'''
        return instance.process_data(self.data) #이미 세팅된 메인데이터를 활용한다.
            
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [5]:
#bitmex = Bitmex(, MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#bitmex.load()

In [10]:
bitmex = Bitmex()
#bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
입력받은 15분봉으로 2022-12-31 15:01:00 부터 표현합니다.
CSV 데이터 로드 완료.
전처리 완료


In [11]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int
0,2023-01-01 00:00:00,16583.5,16583.5,16608.0,16600.0,1.672499e+09
10,2023-01-01 00:15:00,16591.5,16583.5,16608.0,16590.5,1.672500e+09
27,2023-01-01 00:45:00,16590.5,16578.0,16590.5,16580.0,1.672502e+09
47,2023-01-01 01:15:00,16583.0,16573.5,16588.5,16588.5,1.672503e+09
66,2023-01-01 01:45:00,16587.5,16585.5,16596.5,16592.5,1.672505e+09
...,...,...,...,...,...,...
174825,2023-05-08 11:00:00,28308.0,28207.0,28374.0,28362.0,1.683511e+09
174840,2023-05-08 11:15:00,28362.0,28293.0,28374.0,28329.5,1.683512e+09
174855,2023-05-08 11:30:00,28329.5,28260.0,28330.5,28330.5,1.683513e+09
174870,2023-05-08 11:45:00,28330.5,28315.5,28395.0,28376.0,1.683514e+09


In [12]:
indicator_list = [MovingAverageProcessing(), RSIProcessing()]
bitmex.add_sub_indicator(indicator_list)

이동 평균을 계산했습니다.
RSI를 계산했습니다.


In [13]:
indicator_list_more = [HighPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

수행한 숫자:11395
가장 높은 점수:200


In [14]:
indicator_list_more = [LowPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

In [15]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score
0,2023-01-01 00:00:00,16583.5,16583.5,16608.0,16600.0,1.672499e+09,16600.000000,16600.000000,0.000000,0,0
10,2023-01-01 00:15:00,16591.5,16583.5,16608.0,16590.5,1.672500e+09,16595.250000,16595.250000,0.000000,0,0
27,2023-01-01 00:45:00,16590.5,16578.0,16590.5,16580.0,1.672502e+09,16590.166667,16590.166667,0.000000,0,0
47,2023-01-01 01:15:00,16583.0,16573.5,16588.5,16588.5,1.672503e+09,16589.750000,16589.750000,29.824561,0,0
66,2023-01-01 01:45:00,16587.5,16585.5,16596.5,16592.5,1.672505e+09,16590.300000,16590.300000,38.461538,0,0
...,...,...,...,...,...,...,...,...,...,...,...
174825,2023-05-08 11:00:00,28308.0,28207.0,28374.0,28362.0,1.683511e+09,28675.150000,28868.091667,30.264244,0,0
174840,2023-05-08 11:15:00,28362.0,28293.0,28374.0,28329.5,1.683512e+09,28642.475000,28858.633333,30.276745,0,0
174855,2023-05-08 11:30:00,28329.5,28260.0,28330.5,28330.5,1.683513e+09,28614.275000,28850.125000,29.519833,0,0
174870,2023-05-08 11:45:00,28330.5,28315.5,28395.0,28376.0,1.683514e+09,28590.850000,28842.741667,32.255457,0,0


In [36]:
process_instance = GetHighPoints(bitmex.data.loc[bitmex.data['high_score']!=0], 'high_score', 0.95)
high_points = bitmex.get_key_points(process_instance)
high_points

정형화된 기준값은 : 0.6917085427135654


C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:251: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:259: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:259: DeprecationWarning:

In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, 

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,normalized_value_low,origin_high_score,high_for_graph
4639,2023-01-05 04:00:00,16950.5,16861.0,16978.0,16870.0,1.672859e+09,16853.975,16847.491667,62.535211,150.0,0,0.748744,0.0,150,16978.0
6925,2023-01-07 05:00:00,16926.5,16926.5,17050.0,16983.5,1.673035e+09,16820.250,16788.091667,84.920635,151.0,0,0.753769,0.0,151,17050.0
15655,2023-01-14 09:30:00,20370.0,20332.0,21550.0,21081.0,1.673656e+09,19805.425,19313.258333,93.363181,200.0,0,1.000000,0.0,200,21550.0
18589,2023-01-16 11:15:00,21163.5,21164.0,21478.0,21356.5,1.673835e+09,20976.050,20878.308333,73.813421,139.0,0,0.693467,0.0,139,21478.0
22121,2023-01-18 23:15:00,21500.5,21341.0,21652.5,21392.5,1.674051e+09,21252.400,21245.366667,71.482176,200.0,0,1.000000,0.0,200,21652.5
26578,2023-01-22 02:45:00,23272.5,23128.0,23361.0,23220.0,1.674323e+09,23076.975,22846.266667,64.471744,200.0,0,1.000000,0.0,200,23361.0
29440,2023-01-24 03:15:00,23110.0,23071.0,23175.0,23092.0,1.674498e+09,22864.875,22806.516667,79.885057,173.0,0,0.864322,0.0,173,23175.0
32498,2023-01-26 07:00:00,23605.0,23162.0,23903.0,23446.0,1.674684e+09,22848.200,22695.175000,81.957547,200.0,0,1.000000,0.0,200,23903.0
37971,2023-01-30 04:45:00,23929.5,23874.0,23970.5,23907.0,1.675022e+09,23628.600,23436.266667,84.981685,200.0,0,1.000000,0.0,200,23970.5
42530,2023-02-02 09:45:00,24179.5,24066.5,24264.0,24192.0,1.675299e+09,23726.950,23277.483333,76.595745,200.0,0,1.000000,0.0,200,24264.0


In [38]:
process_instance = GetLowPoints(bitmex.data.loc[bitmex.data['low_score']!=0], 'low_score', 0.95)
low_points = bitmex.get_key_points(process_instance)
low_points

정형화된 기준값은 : 0.5537688442211035


C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:295: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:303: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:303: DeprecationWarning:

In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, 

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,normalized_value_low,origin_low_score,low_for_graph
3322,2023-01-04 02:45:00,16608.5,16596.0,16611.5,16611.5,1.672768e+09,16667.400,16688.633333,28.947368,0,123.0,0.000,0.613065,123,16596.0
6573,2023-01-06 22:30:00,16672.0,16665.5,16775.0,16759.5,1.673012e+09,16748.925,16793.658333,46.200000,0,191.0,0.000,0.954774,191,16665.5
22271,2023-01-19 01:45:00,20473.5,20398.0,20922.0,20844.5,1.674060e+09,21194.050,21227.766667,39.247881,0,200.0,0.000,1.000000,200,20398.0
28178,2023-01-23 05:45:00,22515.0,22265.0,22539.0,22459.5,1.674420e+09,22769.300,22838.300000,18.754864,0,185.0,0.000,0.924623,185,22265.0
31271,2023-01-25 10:15:00,22584.5,22333.5,22585.5,22364.5,1.674609e+09,22750.700,22871.416667,26.251097,0,200.0,0.000,1.000000,200,22333.5
34137,2023-01-27 10:30:00,22887.0,22525.0,22904.5,22672.0,1.674783e+09,22985.900,23017.550000,16.161616,0,127.0,0.000,0.633166,127,22525.0
39561,2023-01-31 07:30:00,22696.5,22368.5,22706.5,22699.5,1.675118e+09,22824.350,23090.491667,43.135518,0,200.0,0.000,1.000000,200,22368.5
42215,2023-02-02 04:30:00,22987.5,22690.5,23306.0,23269.5,1.675280e+09,23034.500,23051.283333,68.204758,0,172.0,0.000,0.859296,172,22690.5
44748,2023-02-03 23:15:00,23323.5,23178.0,23354.0,23304.5,1.675434e+09,23440.625,23477.808333,34.362934,0,135.0,0.000,0.673367,135,23178.0
48271,2023-02-06 14:15:00,22874.0,22624.0,22878.5,22705.0,1.675660e+09,22939.350,22959.591667,21.576763,0,200.0,0.000,1.000000,200,22624.0


In [39]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(bitmex.data, high_points, low_points)
draw_instance.visualize()

In [ ]:
bitmex.data['high_score']

CSV 데이터를 로드하고 Nan을 제거합니다.
입력받은 60분봉으로 2022-12-31 15:01:00 부터 표현합니다.
CSV 데이터 로드 완료.
전처리 완료


In [27]:
bitmex_60.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high
21,2023-01-01 01:00:00,16603.5,16573.5,16603.5,16592.5,1.672502e+09,16592.500000,16592.500000,0.000000,0,0,0.0
146,2023-01-01 05:00:00,16562.0,16551.5,16565.5,16551.5,1.672517e+09,16572.000000,16572.000000,0.000000,0,0,0.0
182,2023-01-01 06:00:00,16554.0,16540.0,16554.0,16540.0,1.672520e+09,16561.333333,16561.333333,0.000000,0,0,0.0
238,2023-01-01 07:00:00,16540.0,16424.5,16553.5,16502.5,1.672524e+09,16546.625000,16546.625000,0.000000,0,4,0.0
293,2023-01-01 08:00:00,16502.0,16472.5,16530.0,16522.0,1.672528e+09,16541.700000,16541.700000,17.808219,0,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
174585,2023-05-08 07:00:00,28860.0,28785.5,28916.0,28807.0,1.683497e+09,28933.750000,29124.300000,39.591568,0,0,0.0
174645,2023-05-08 08:00:00,28807.0,28465.5,28860.0,28497.0,1.683500e+09,28910.750000,29114.675000,29.489696,0,0,0.0
174705,2023-05-08 09:00:00,28497.0,28424.0,28690.0,28629.5,1.683504e+09,28898.250000,29107.158333,39.132399,0,0,0.0
174765,2023-05-08 10:00:00,28629.5,28126.0,28652.5,28308.0,1.683508e+09,28870.075000,29093.741667,28.979300,0,0,0.0


이동 평균을 계산했습니다.
RSI를 계산했습니다.


In [31]:
bitmex_60.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,normalized_value_low
21,2023-01-01 01:00:00,16603.5,16573.5,16603.5,16592.5,1.672502e+09,16592.500000,16592.500000,0.000000,0,0,0.0,0.000
146,2023-01-01 05:00:00,16562.0,16551.5,16565.5,16551.5,1.672517e+09,16572.000000,16572.000000,0.000000,0,0,0.0,0.000
182,2023-01-01 06:00:00,16554.0,16540.0,16554.0,16540.0,1.672520e+09,16561.333333,16561.333333,0.000000,0,0,0.0,0.000
238,2023-01-01 07:00:00,16540.0,16424.5,16553.5,16502.5,1.672524e+09,16546.625000,16546.625000,0.000000,0,4,0.0,0.020
293,2023-01-01 08:00:00,16502.0,16472.5,16530.0,16522.0,1.672528e+09,16541.700000,16541.700000,17.808219,0,1,0.0,0.005
...,...,...,...,...,...,...,...,...,...,...,...,...,...
174585,2023-05-08 07:00:00,28860.0,28785.5,28916.0,28807.0,1.683497e+09,28933.750000,29124.300000,39.591568,0,0,0.0,0.000
174645,2023-05-08 08:00:00,28807.0,28465.5,28860.0,28497.0,1.683500e+09,28910.750000,29114.675000,29.489696,0,0,0.0,0.000
174705,2023-05-08 09:00:00,28497.0,28424.0,28690.0,28629.5,1.683504e+09,28898.250000,29107.158333,39.132399,0,0,0.0,0.000
174765,2023-05-08 10:00:00,28629.5,28126.0,28652.5,28308.0,1.683508e+09,28870.075000,29093.741667,28.979300,0,0,0.0,0.000


In [49]:
bitmex_60 = Bitmex()
#bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex_60.set_loader(BitmexCSVDataLoader(60,'2022-12-31 15:01:00'))
bitmex_60.load()

indicator_list_60 = [MovingAverageProcessing(), RSIProcessing()]
bitmex_60.add_sub_indicator(indicator_list_60)

indicator_list_more_60 = [HighPointScoringProcessing([48])]
bitmex_60.add_sub_indicator(indicator_list_more_60)

indicator_list_more_60 = [LowPointScoringProcessing([48])]
bitmex_60.add_sub_indicator(indicator_list_more_60)




CSV 데이터를 로드하고 Nan을 제거합니다.
입력받은 60분봉으로 2022-12-31 15:01:00 부터 표현합니다.
CSV 데이터 로드 완료.
전처리 완료
이동 평균을 계산했습니다.
RSI를 계산했습니다.
수행한 숫자:2892
가장 높은 점수:48


In [50]:

process_instance_60 = GetHighPoints(bitmex_60.data.loc[bitmex_60.data['high_score']!=0], 'high_score', 0.80)
high_points_60 = bitmex_60.get_key_points(process_instance_60)

process_instance_60 = GetLowPoints(bitmex_60.data.loc[bitmex_60.data['low_score']!=0], 'low_score', 0.80)
low_points_60 = bitmex_60.get_key_points(process_instance_60)

정형화된 기준값은 : 0.1702127659574468
정형화된 기준값은 : 0.1829787234042558


C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:251: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:259: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\User\AppData\Local\Temp/ipykernel_11700/483221344.py:259: DeprecationWarning:

In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, 

In [51]:
draw_instance = BasicPriceWithRsiVisualization()
draw_instance.set_data(bitmex_60.data, high_points_60, low_points_60)
draw_instance.visualize()